# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:25<00:00,  8.45s/it]


In [3]:
len(deals)

30

In [4]:
deals[3].describe()

"Title: X96 Mini 8GB Android 4K Smart TV Box for $16 + free shipping w/ first order\nDetails: That's the lowest price we could find by $14 after factoring in the $6.89 import charges and 75-cent processing fee.New Alibaba customers get free shipping; otherwise, it starts at around $10. Buy Now at Alibaba\nFeatures: \nURL: https://www.dealnews.com/X96-Mini-8-GB-Android-4-K-Smart-TV-Box-for-16-free-shipping-w-first-order/21806852.html?iref=rss-c142"

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [5]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [6]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [8]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
# print(user_prompt[:2000])
print(user_prompt)
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Refurb Apple iPhone 15 256GB Smartphone for $386 + free shipping
Details: That's $514 off and the best deal we've seen on this model. A 1-year Allstate warranty applies. Buy Now at eBay
Features: 
URL: https://www.dealnews.com/Refurb-Apple-iPhone-15-256-GB-Smartphone-for-386-free-shipping/21806918.html?iref=rss-c142

Title: Best Buy Outlet Event: Up to 60% off + free shipping
D

In [9]:
DealSelection.model_json_schema()

{'$defs': {'Deal': {'description': 'A class to Represent a Deal with a summary description',
   'properties': {'product_description': {'description': "Your clearly expressed summary of the product in 3-4 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a short paragraph of text for each item you choose.",
     'title': 'Product Description',
     'type': 'string'},
    'price': {'description': 'The actual price of this product, as advertised in the deal. Be sure to give the actual price; for example, if a deal is described as $100 off the usual $300 price, you should respond with $200',
     'title': 'Price',
     'type': 'number'},
    'url': {'description': 'The URL of the deal, as provided in the input',
     'title': 'Url',
     'type': 'string'}},
   'required': ['product_description', 'price', 'url'],
   'title': 'Deal',
   'type': 'object'}},
 'description': 'A clas

In [10]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='Apple Mac mini (2024) with the M4 chip is a compact desktop powered by Apple Silicon offering modern performance for macOS workflows. This configuration includes 16GB of unified RAM and a 256GB solid-state drive, suitable for general productivity, media tasks, and light creative work. The unit runs macOS and features the I/O and efficiency benefits of the M4 architecture in a small-footprint aluminum chassis.', price=400.0, url='https://www.dealnews.com/products/Apple/Apple-Mac-mini-M4-Desktop-2024-w-256-GB-SSD/484139.html?iref=rss-c39'), Deal(product_description='Onn 65" 4K HDR LED UHD Smart TV is a 65-inch 2160p Roku TV that supports HDR content and integrates with Apple Home, Amazon Alexa, and Google Home. It includes three HDMI ports for connecting consoles, streaming devices, and Blu-ray players, and runs the Roku smart TV platform for apps and streaming services. The large-screen panel is aimed at living-room viewing for movies, spor

In [11]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


Apple Mac mini (2024) with the M4 chip is a compact desktop powered by Apple Silicon offering modern performance for macOS workflows. This configuration includes 16GB of unified RAM and a 256GB solid-state drive, suitable for general productivity, media tasks, and light creative work. The unit runs macOS and features the I/O and efficiency benefits of the M4 architecture in a small-footprint aluminum chassis.
400.0
https://www.dealnews.com/products/Apple/Apple-Mac-mini-M4-Desktop-2024-w-256-GB-SSD/484139.html?iref=rss-c39

Onn 65" 4K HDR LED UHD Smart TV is a 65-inch 2160p Roku TV that supports HDR content and integrates with Apple Home, Amazon Alexa, and Google Home. It includes three HDMI ports for connecting consoles, streaming devices, and Blu-ray players, and runs the Roku smart TV platform for apps and streaming services. The large-screen panel is aimed at living-room viewing for movies, sports, and gaming.
278.0
https://www.dealnews.com/products/Onn/Onn-65-4-K-HDR-LED-UHD-Smart-

In [12]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [13]:
from agents.scanner_agent import ScannerAgent

In [14]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [15]:
result

DealSelection(deals=[Deal(product_description='Refurbished Apple iPhone 15 with 256GB of internal storage, offering the standard iPhone 15 feature set in a refurbished condition. This model includes modern Apple hardware and storage ample for apps, photos, and video; it ships with a 1-year Allstate warranty which covers defects for a year. The listing is aimed at buyers who want current-generation iPhone performance at a reduced price compared with new units.', price=386.0, url='https://www.dealnews.com/Refurb-Apple-iPhone-15-256-GB-Smartphone-for-386-free-shipping/21806918.html?iref=rss-c142'), Deal(product_description='Plaud NotePin AI Note Taker — a wearable, voice-focused recording device with advanced on-device AI transcription and summarization. It records continuously for up to about 20 hours, supports multimodal input, and is designed to create searchable notes and summarized transcripts for meetings, interviews, and lectures. The device is compact and intended for hands-free c

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [16]:
load_dotenv(override=True)

True

In [17]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [18]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [20]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [21]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [22]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [23]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message
12:21:25 - LiteLLM:INFO: utils.py:3872 - 
LiteLLM completion() model= claude-sonnet-4-5; provider = anthropic
INFO:LiteLLM:
LiteLLM completion() model= claude-sonnet-4-5; provider = anthropic
12:21:27 - LiteLLM:INFO: utils.py:1621 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
